In [72]:
import json
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd

def extract_pattern_pairs(log_path: str) -> List[Dict[str, Any]]:
    """Parse a summarization log and return pattern name collections per cluster."""
    lines = Path(log_path).read_text(encoding="utf-8").splitlines()

    entries: List[Dict[str, Any]] = []
    cluster_id: int | None = None
    section: str | None = None
    buffer: List[str] = []
    original_names: List[str] = []
    summarized_names: List[str] = []
    thinkings: str = ""

    def flush(active_section: str | None) -> None:
        nonlocal buffer, original_names, summarized_names, thinkings
        if not active_section:
            buffer = []
            return
        raw = "\n".join(buffer).strip()
        buffer = []
        if not raw:
            return
        if active_section == "original":
            try:
                data = json.loads(raw)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Failed to parse original patterns for cluster {cluster_id}") from exc
            original_names = [item.get("Pattern Name") for item in data if isinstance(item, dict)]
        elif active_section == "summary":
            try:
                data = json.loads(raw)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Failed to parse summarized patterns for cluster {cluster_id}") from exc
            summarized_names = [
                item.get("Pattern Name")
                for item in data.get("patterns", [])
                if isinstance(item, dict)
            ]
            thinkings = data.get("thinkings","")

    for line in lines:
        stripped = line.strip()
        if stripped.startswith("Cluster ") and stripped.split()[1].isdigit():
            flush(section)
            if cluster_id is not None:
                entries.append({
                    "cluster": cluster_id,
                    "original_patterns": original_names,
                    "summarized_patterns": summarized_names,
                    "thinking": thinkings,
                })
            cluster_id = int(stripped.split()[1])
            section = None
            original_names = []
            summarized_names = []
        elif stripped == "Original Patterns:":
            flush(section)
            section = "original"
        elif stripped == "Summarized Patterns:":
            flush(section)
            section = "summary"
        elif stripped == "Verification Result:":
            flush(section)
            section = None
        elif section:
            buffer.append(line)

    flush(section)
    if cluster_id is not None:
        entries.append({
            "cluster": cluster_id,
            "original_patterns": original_names,
            "summarized_patterns": summarized_names,
            "thinking": thinkings,
        })

    return entries

In [76]:
log_file = "./logs/summarization_log_iter_01.txt"
pattern_pairs_01 = extract_pattern_pairs(log_file)
pattern_pairs_df_01 = pd.DataFrame(pattern_pairs_01)
iter_01_summarized_patterns = set(pattern_pairs_df_01["summarized_patterns"].explode())

In [77]:
log_file = "./logs/summarization_log_iter_02.txt"
pattern_pairs_02 = extract_pattern_pairs(log_file)
pattern_pairs_df_02 = pd.DataFrame(pattern_pairs_02)
iter_02_summarized_patterns = set(pattern_pairs_df_02["summarized_patterns"].explode())

In [78]:
log_file = "./logs/summarization_log_iter_03.txt"
pattern_pairs_03 = extract_pattern_pairs(log_file)
pattern_pairs_df_03 = pd.DataFrame(pattern_pairs_03)
iter_03_summarized_patterns = set(pattern_pairs_df_03["summarized_patterns"].explode())

In [74]:
pattern_pairs_01

[{'cluster': 0,
  'original_patterns': ['External Knowledge Augmentation',
   'Domain-Specific Tool Integration',
   'Multimodal Interaction Augmentation',
   'Knowledge Conflict Resolution (in Tool Augmentation)',
   'Tool-Augmented Foundation Model',
   'Tool Augmentation',
   'Retrieval Augmentation',
   'Knowledge Augmentation',
   'LLM-based Tool Learning',
   'Modular Reasoning, Knowledge and Language (MRKL) System'],
  'summarized_patterns': ['LLM Tool Orchestration'],
  'thinking': "The provided cluster of design patterns consistently addresses the limitations of Large Language Models (LLMs) by augmenting their capabilities with external tools and knowledge sources. While the individual patterns highlight specific aspects—such as accessing external knowledge, integrating domain-specific tools, handling multimodal inputs, or resolving knowledge conflicts—they all share the same fundamental underlying principle: using an LLM as an intelligent orchestrator to interact with and lev

In [75]:
pattern_pairs_df_01

,cluster,original_patterns,summarized_patterns,thinking
0,0,"[External Knowledge Augmentation, Domain-Speci...",[LLM Tool Orchestration],The provided cluster of design patterns consis...
1,1,"[Tool Use / Tool Augmentation, Task Automation...",[Tool-Augmented Agent],"The two provided patterns, 'Tool Use / Tool Au..."
2,2,"[Trust Calibration through Transparency, Inher...","[Explainable AI (XAI) Techniques, AI System Pr...","The initial cluster contained six patterns, al..."
3,3,"[Prompt-based Defenses, Robust Tool-Augmented ...","[In-Prompt Guardrails, External Augmentation a...","I analyzed the two provided patterns, 'Prompt-..."
4,4,"[Iterative Task Solving (with Feedback), Multi...",[Iterative Self-Correction with Feedback],I analyzed the provided cluster of five design...
...,...,...,...,...
72,72,"[LLMs as Personalized Content Creator, LLMs as...","[LLM as Intelligent Processor & Reasoner, LLM ...",The initial cluster contained eight distinct p...
73,73,"[Live Web Access with Controlled Interaction, ...",[Agentic Web Interaction Layer],"The two provided patterns, 'Live Web Access wi..."
74,74,"[Rejection Sampling (Best-of-N), KL Regulariza...",[Reward Model Guided Alignment],The cluster contained three patterns:\n1. **R...
75,75,"[Retriever-Aware Training (RAT), AST-based Hal...","[Dynamic Tool-Augmented LLM Framework, Structu...",The initial cluster contained six patterns rel...


In [15]:
intercection_patterns = iter_01_summarized_patterns & iter_02_summarized_patterns# & iter_03_summarized_patterns
print(f"Number of patterns common in all 3 iterations: {len(intercection_patterns)}")
print("Common patterns:")
for pattern in intercection_patterns:
    print(pattern)

Number of patterns common in all 3 iterations: 26
Common patterns:
Denoising Pretraining for Foundational Generative Models
Nearest Neighbor Output Augmentation
Augmented Response Synthesis
Exploratory Reasoning (Tree/Graph of Thoughts)
Retrieval-Augmented Generation (RAG)
Autonomous Tool Generation
Progressive Response Disclosure
Reasoning-Action Alignment
LLM Fallback to Inherent Knowledge
Direct LLM Generation
RAG KV Cache Optimization System
Structured Output Generation
Parameter-Efficient LLM Adaptation
Task Conditioning with Control Tokens
Personalized Tool Interaction
AI System Process Transparency and Trust Calibration
User Intent Resolution
LLM-Guided Human Prompt Refinement
Contextual Refinement
Explicit Step-by-Step Reasoning (Chain-of-Thought)
Adversarial Robustness Evaluation
Ambiguity-Robust Demonstrations
In-Prompt Guardrails
Efficient Dense Semantic Retrieval
Proactive Static Planning
Persona and Contextual Framing


In [ ]:
pattern_pairs_df_01["original_patterns"] = pattern_pairs_df_01["original_patterns"].apply(lambda x: str([f"{i+1} {row}" for i,row in enumerate(x)]).replace("[","").replace("]","").replace(",","\n").replace("'",""))
pattern_pairs_df_02["original_patterns"] = pattern_pairs_df_02["original_patterns"].apply(lambda x: str([f"{i+1} {row}" for i,row in enumerate(x)]).replace("[","").replace("]","").replace(",","\n").replace("'",""))
pattern_pairs_df_03["original_patterns"] = pattern_pairs_df_03["original_patterns"].apply(lambda x: str([f"{i+1} {row}" for i,row in enumerate(x)]).replace("[","").replace("]","").replace(",","\n").replace("'",""))

pattern_pairs_df_01["summarized_patterns"] = pattern_pairs_df_01["summarized_patterns"].apply(lambda x: str([f"{i+1} {row}" for i,row in enumerate(x)]).replace("[","").replace("]","").replace(",","\n").replace("'",""))
pattern_pairs_df_02["summarized_patterns"] = pattern_pairs_df_02["summarized_patterns"].apply(lambda x: str([f"{i+1} {row}" for i,row in enumerate(x)]).replace("[","").replace("]","").replace(",","\n").replace("'",""))
pattern_pairs_df_03["summarized_patterns"] = pattern_pairs_df_03["summarized_patterns"].apply(lambda x: str([f"{i+1} {row}" for i,row in enumerate(x)]).replace("[","").replace("]","").replace(",","\n").replace("'",""))

In [80]:
pattern_pairs_df_01[["original_patterns","summarized_patterns","thinking"]].to_csv("./summarized_patterns/comparison/iter_01_patterns.csv", index=False)
pattern_pairs_df_02[["original_patterns","summarized_patterns","thinking"]].to_csv("./summarized_patterns/comparison/iter_02_patterns.csv", index=False)
pattern_pairs_df_03[["original_patterns","summarized_patterns","thinking"]].to_csv("./summarized_patterns/comparison/iter_03_patterns.csv", index=False)

In [83]:
for col in ["original_patterns","summarized_patterns","thinking"]:
    pattern_pairs_df_01[col] = pattern_pairs_df_01[col].apply(lambda x: str(x).replace("\n","<br>"))
    pattern_pairs_df_02[col] = pattern_pairs_df_02[col].apply(lambda x: str(x).replace("\n","<br>"))
    pattern_pairs_df_03[col] = pattern_pairs_df_03[col].apply(lambda x: str(x).replace("\n","<br>"))

pattern_pairs_df_01[["original_patterns","summarized_patterns","thinking"]].to_markdown("./summarized_patterns/comparison/iter_01_patterns.md",)
pattern_pairs_df_02[["original_patterns","summarized_patterns","thinking"]].to_markdown("./summarized_patterns/comparison/iter_02_patterns.md", index=False)
pattern_pairs_df_03[["original_patterns","summarized_patterns","thinking"]].to_markdown("./summarized_patterns/comparison/iter_03_patterns.md", index=False)